In [1]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/Project/'
netCDF=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
true_time=netCDF['time']
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
times=netCDF['time'].values/(1e9 * 60); times=times.astype(float);

#Restricts the timesteps of the data from timesteps0 to 140
data=netCDF.isel(time=np.arange(0,140+1))
parcel=parcel.isel(time=np.arange(0,140+1))


# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# parcel=parcel.isel(time=np.arange(0,400+1))

In [ ]:
###########################
# Equivalent Potential Temperature (theta_e)
# surface_prs=data['prs'].mean(dim=("time",'yh','xh')); surface_prs=surface_prs[0]

################################# PRESSURE VARIABLES
p0=1e5
P=data['prs'].data

################################# MIXING RATIO
qv=data['qv'].data
qt=(data['qv']+data['qc']+data['qr']+data['qi']+data['qs']+data['qg']).data #total mixing ratio

################################# THERMODYNAMICS
Rd=287.04
Rv=461.5
Cpd=1005.7 #+-2.5
Cpv=1870 #+-25
Cpl=4190 #+-30
Lv0=2.501e6
def Lv(T): #Kirchoff's formula L_i,ii= L_i,ii0+(Cpii-Cpi)*(T-273.15)
    Llv=Lv0+(Cpv-Cpl)*(T-273.15) #should it be Cpl. is Cl the same?***
    return Llv

################################# TEMPERATURE
theta=data['th'].data
T=theta*(P/p0)**(Rd/Cpd)

################################# RELATIVE HUMIDITY
eps=0.622
#qv=eps*(e/(P-e)) ==> e = qv*P/(eps+qv)
e=qv*P/(qv+eps)
Pd=P-e #P=Pd+e ==> Pd=P-e

e_s0=611
T0=273.15
inner=(Lv(T)/Rv)*((1/T0)-(1/T))
e_s=e_s0*np.exp(inner)
H = e/e_s ########



In [ ]:
LV=Lv(T)
mean=np.mean(LV,axis=(2,3))
plt.contourf(mean.T,levels=50)
plt.colorbar();
plt.title(r'$L_v$');

In [ ]:
divisor=(Cpd+Cpl*qt)
theta_e_approx=theta*np.exp(Lv(T)*qv/(divisor*T))
mean=np.mean(theta_e_approx,axis=(2,3))
plt.contourf(mean.T,levels=50)
plt.colorbar();
plt.title('Approximate ' + r'$\theta_e$');

In [ ]:
mean=np.mean(e_s,axis=(2,3))
plt.contourf(mean.T,levels=50)
plt.colorbar();
plt.title(r'$e_s$');

In [ ]:
divisor=(Cpd+Cpl*qt)
theta_e=(T*(p0/Pd)**(Rd/divisor))*(H**(-qv*Rv/divisor))*np.exp(Lv(T)*qv/(divisor*T))

mean=np.mean(theta_e,axis=(2,3))
plt.contourf(mean.T,levels=50)
plt.colorbar();
plt.title('Non-approximate ' + r'$\theta_e$');

In [ ]:
hey=np.mean((theta_e-theta_e_approx),axis=(2,3))
plt.contourf(hey.T,levels=50)
plt.colorbar()
plt.title('comparing non-approximate minus approximate ' + r'$\theta_e$')

In [ ]:
vert_mean1=np.mean(theta_e_approx,axis=(0,2,3))
vert_mean2=np.mean(theta_e,axis=(0,2,3))

plt.plot(vert_mean1,data['zh'],label='approx')
plt.plot(vert_mean2,data['zh'],label='non-approx')
plt.legend();
plt.title('comparing approx and non-approximate ' + r'$\theta_e$') #most difference at the surface 
#conclusion: use non-approximated version 
plt.xlim((320,380))

In [ ]:
with h5py.File(dir+'theta_e_approx.h5', 'w') as f:
    # Save the array as a variable in the file
    f.create_dataset('theta_e_approx', data=theta_e_approx)

with h5py.File(dir+'theta_e.h5', 'w') as f:
    # Save the array as a variable in the file
    f.create_dataset('theta_e', data=theta_e)